In [ ]:
import pandas as pd

pop=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\poplprofile_v2_clean.csv")
gap=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores.csv")
print("poplprofile_v2_clean.csv shape:",pop.shape)
print("gap_scores.csv shape:",gap.shape,"(32 boroughs )")

In [ ]:
eth_2223=pop[(pop['demographic_group'] == 'Eth7') &(pop['geography_level'] == 'borough') &(pop['survey_year'] == '2022-23')].copy()

print(f"2022-23 Eth7 x borough rows  {len(eth_2223)}")
print(f"  {(eth_2223['suppress'] == True).sum()} " f"({100 * (eth_2223['suppress'] == True).mean():.1f}%)")
print(f" at least one non suppress: "f"{eth_2223[~eth_2223['suppress']]['borough'].nunique()} / {gap['borough'].nunique()}")
cats_per_borough_2223=eth_2223[~eth_2223['suppress']].groupby('borough')['category'].nunique()
print(f"\nNon suppresse categories per borough only 2022-23  " f"(7 categories):")
print(cats_per_borough_2223.describe())
print(f"\n 1-2 non- sppressed categories in 2022-23 "f"{(cats_per_borough_2223 <= 2).sum()} / {len(cats_per_borough_2223)}")

In [ ]:
eth_all=pop[(pop['demographic_group'] == 'Eth7') &(pop['geography_level'] == 'borough') &(~pop['suppress'])].copy()
print(f" {len(eth_all)}")
print(f"boroughs {eth_all['borough'].nunique()} / {gap['borough'].nunique()}")
eth_summary=eth_all.groupby(['borough','category']).agg(group_inactive=('pct_inactive','mean'),total_respondents=('respondents','sum'),n_years_pooled=('survey_year','nunique'),).reset_index()
print(f"borough x ethnic-group cells {len(eth_summary)}")
print(f"categories per borough for all years")
print(eth_summary.groupby('borough')['category'].nunique().describe())

In [ ]:
#equity gap table 
ethnicity_gap=eth_summary.merge(gap[['borough','inactive']].rename(columns={'inactive': 'borough_overall_inactive'}),on='borough',how='inner')
ethnicity_gap['gap']=ethnicity_gap['group_inactive'] - ethnicity_gap['borough_overall_inactive']
ethnicity_gap=ethnicity_gap[['borough','category','group_inactive','borough_overall_inactive','gap','total_respondents','n_years_pooled']].rename(columns={'category': 'ethnic_group'})

pd.set_option('display.max_rows',250)
print(f" {len(ethnicity_gap)} borough x ethnic-group rows, "f"{ethnicity_gap['borough'].nunique()} boroughs")
print(ethnicity_gap.sort_values(['borough','ethnic_group']).to_string(index=False))

In [ ]:
#rank for largest gap
idx_max_abs=ethnicity_gap.groupby('borough')['gap'].apply(lambda s: s.abs().idxmax())
worst_per_borough=ethnicity_gap.loc[idx_max_abs].sort_values('gap',key=abs,ascending=False)
print("Largest gap per borough")
print(worst_per_borough[['borough','ethnic_group','group_inactive','borough_overall_inactive','gap','total_respondents','n_years_pooled']].head(15).to_string(index=False))

In [ ]:
print("group more inactive that average in borough")
print(ethnicity_gap.sort_values('gap',ascending=False)[['borough','ethnic_group','group_inactive','borough_overall_inactive','gap','total_respondents']].head(10).to_string(index=False))
print("group less inactive that average in borough")
print(ethnicity_gap.sort_values('gap',ascending=True)[['borough','ethnic_group','group_inactive','borough_overall_inactive','gap','total_respondents']].head(10).to_string(index=False))

In [ ]:
gap_equity=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")

print("IMD equity_gap ")
print(gap_equity['equity_gap'].describe())
print("Ethnicity equity gap ")
print(worst_per_borough['gap'].abs().describe())


In [ ]:

OUT_PATH=r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\ethnicity_equity_gaps.csv"
export_df=ethnicity_gap.copy()

#reliable if respindents are more than 200
export_df['reliable']=export_df['total_respondents'] >= 200
export_df=export_df.sort_values('gap',ascending=False).reset_index(drop=True)
export_df=export_df[['borough','ethnic_group','group_inactive','borough_overall_inactive','gap','total_respondents','n_years_pooled','reliable']]
export_df.to_csv(OUT_PATH,index=False)

print(f"{len(export_df)} to {OUT_PATH}")
print(f"\nReliable: {export_df['reliable'].sum()}")
print(f"Unreliable  {(~export_df['reliable']).sum()}")
print(export_df.head(15).to_string(index=False))

In [ ]:
rec=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\intervention_recommendations.csv")
under=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\under_monitored_flags.csv")
print("rec")
print(rec.to_string(index=False))
print("\nnot monitored enough")
print(under.to_string(index=False))

In [ ]:
df=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\individual_level_london.csv")
london_avg_opp=df['readiness_opportunity'].mean()
london_avg_ab=df['readiness_ability'].mean()
borough_readiness=df.groupby('borough').agg(avg_opportunity=('readiness_opportunity','mean'),avg_ability=('readiness_ability','mean')).reset_index()
borough_readiness['opportunity_gap']=london_avg_opp - borough_readiness['avg_opportunity']
borough_readiness['ability_gap']=london_avg_ab - borough_readiness['avg_ability']
borough_readiness['dominant_barrier']=borough_readiness.apply(lambda r: 'opportunity' if r['opportunity_gap'] > r['ability_gap'] else 'ability',axis=1)

target=['Redbridge','Enfield','Hammersmith and Fulham','Greenwich','Lambeth','Bromley','Richmond upon Thames']
print(borough_readiness[borough_readiness['borough'].isin(target)][['borough','avg_opportunity','avg_ability','opportunity_gap','ability_gap','dominant_barrier']].sort_values('opportunity_gap',ascending=False).to_string(index=False))
print(f"\n avg opportunity: {london_avg_opp:.3f}")
print(f" avg ability: {london_avg_ab:.3f}")

In [ ]:
df=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
print(df[df['borough'].isin(['Enfield','Redbridge','Barking and Dagenham'])][['borough','class','note']])

In [ ]:
df=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
for _,row in df[df['borough'].isin(['Enfield','Redbridge','Barking and Dagenham'])].iterrows():
    print(f"\n{row['borough']} ({row['class']}):")
    print(row['note'])